# DOUBLE Q-LEARNING - SUTTON

This small modification is meant to solve the maximisation bias from traditional Q-Learnig

![alt text](../static/dql-diagram-sutton.png "Title")

![alt text](../static/dql-pseudo-code-sutton.png "Title")

In [ ]:
from collections import defaultdict
from IPython import display
import gym
import matplotlib.pyplot as plt
import numpy as np

from src.utils.plots import plot_training_results

%matplotlib inline

In [ ]:
class DoubleQLearning():

    def __init__(self, env, gamma=0.99, epsilon=0.1, alpha=0.5, 
                 ranges=None, grid_step=20, init_factor = 0.001):
        
        self.env = env
        na = env.action_space.n
        self.Qvalues = {
            'Q1': defaultdict(lambda: np.random.randn(na) * init_factor),
            'Q2': defaultdict(lambda: np.random.randn(na) * init_factor)
        }
        self.alpha = alpha 
        self.epsilon = epsilon 
        self.gamma = gamma 
        self.ranges = ranges
        self.grid_step = grid_step        
        
    def encode(self, s):
        if self.ranges is not None:
            to_tuple = np.round(np.divide(s, self.ranges / self.grid_step))
            return tuple(to_tuple.astype(int))
        else:
            return s
            
    def getQs(self, s, q_key):
        return self.Qvalues[q_key][s]

    def act(self, s):
        if np.random.rand() > self.epsilon:
            a = self.act_no_explore(s)
        else: 
            a = self.env.action_space.sample()
        return a

    def act_no_explore(self, s):
        q1, q2 = self.Qvalues.keys()
        return np.argmax(self.Qvalues[q1][s] + self.Qvalues[q2][s])
                    
    def learn(self, s, a, r, s1, done=False):
        target = r
        q_update, q_target = self.Qvalues.keys()

        if np.random.rand() >= 0.5:
            q_update, q_target = q_target, q_update

        if not done:
            max_a = np.argmax(self.Qvalues[q_update][s1]) 
            target = r + self.gamma * self.Qvalues[q_target][s1][max_a]

        qsa = self.Qvalues[q_update][s][a]
        self.Qvalues[q_update][s][a] = (1 - self.alpha) * qsa + self.alpha * target

        return (target - qsa) ** 2

In [ ]:
env = gym.make('CartPole-v1')
env.reset()

samples_s = []
for episode in range(500):
    env.reset()
    while True:
        a = env.action_space.sample()  
        s1, *_, done = env.step(a)
        samples_s.append(s1)
        if done:
            break

grid_range = (np.percentile(samples_s, 90, axis=0) - np.percentile(samples_s, 10, axis=0))

agent = DoubleQLearning(
    env, ranges=grid_range, grid_step=50,
    gamma=0.995, epsilon=1, alpha=0.05
)

nepisodes = 10000
n_log = 1000

min_epsilon = 0.05
epsilon_decay = (1 - min_epsilon) / (nepisodes * 0.9)
history_rewards = []
history_losses = []
for episode in range(nepisodes):
    s = env.reset()
    s = agent.encode(s)
    step = 0
    episode_losses = []
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        s1 = agent.encode(s1)
        loss = agent.learn(s, a, r, s1, done)
        episode_losses.append(loss)
        step += 1
        s = s1
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            mse = np.mean(episode_losses)
            history_rewards.append(step)
            history_losses.append(mse)
            if not episode % n_log:
                print(
                    f'Episode: {episode}, '
                    f'max_step: {np.round(np.mean(history_rewards[-n_log:]), 2)}, '
                    f'mse: {np.round(np.mean(history_losses[-n_log:]), 2)}, '
                    f'eps: {np.round(agent.epsilon, 2)}'
                )
            break

In [ ]:
plot_training_results(
    rewards=history_rewards,
    losses=history_losses,
    window=100,
    scale="log"
)

In [ ]:
env = gym.make('CartPole-v1')
s = env.reset()
img = plt.imshow(env.render(mode='rgb_array')) # Only call this once
for step in range(500):
    s = agent.encode(s)
    img.set_data(env.render(mode='rgb_array')) # Just update the data
    plt.title(f'Step: {step}')
    plt.axis('off')
    display.display(plt.gcf())
    display.clear_output(wait=True)
    action = agent.act_no_explore(s)
    s, _, done, _ = env.step(action)
    if done:
        break
env.close()